# Data Splitting — NPL Early Warning Model

Splits the cleaned feature store into fixed train / test files and saves them to `data_spliting/`, so every
notebook (`iv_define.ipynb`, `xgboot.ipynb`) can load the exact
same rows instead of each re-splitting on its own — that keeps model comparisons fair and reproducible.

Split: 80% train / 20% test, stratified on `IS_NPL` so the NPL rate is preserved in both splits,
`random_state=42` for reproducibility. `xgboost.ipynb` carves its own internal validation set out of the
train file (for early stopping and calibration) so test stays an untouched final holdout.

Run the blocks top to bottom.

## Block 1 — Load the full dataset

In [ ]:
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
TARGET = 'IS_NPL'

DATA_PATH = Path('feature_selection_output/selected_features_xgboost_cleaned.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('C:\\Users\\koemhort.leng\\Desktop\\EWS_NPL_V2\\EDA\\feature_selection_output\\selected_features_xgboost_cleaned.csv')

OUTPUT_DIR = Path('data_spliting')
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH)
print('Shape:', df.shape)
print('Output folder:', OUTPUT_DIR.resolve())

## Block 2 — Check the target

Same guard used in every other notebook in this project: `IS_NPL` must already be 0/1 before we split on it.

In [ ]:
if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError('IS_NPL must be coded as 0 and 1 before splitting.')

print(df[TARGET].value_counts(dropna=False))
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

## Block 3 — Stratified 80/20 split

`stratify=df[TARGET]` keeps the NPL rate consistent across train/test — without it, a plain random
split could accidentally put more (or fewer) NPL accounts in one split by chance.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE)

print(f'Train: {train_df.shape}')
print(f'Test:  {test_df.shape}')

## Block 4 — Verify the split before saving

Confirms row counts add up and the NPL rate held steady across both splits.

In [ ]:
summary = pd.DataFrame({
    'rows': [len(train_df), len(test_df), len(df)],
    'npl_rate_pct': [
        round(train_df[TARGET].mean() * 100, 2),
        round(test_df[TARGET].mean() * 100, 2),
        round(df[TARGET].mean() * 100, 2),
    ],
}, index=['train', 'test', 'full_data'])

assert len(train_df) + len(test_df) == len(df), 'Row counts do not add up to the full dataset'
display(summary)

## Block 5 — Save the splits

Writes `train.csv`, `test.csv`. These are full-column snapshots (no feature
selection applied) — feature selection and encoding stay separate notebook-specific steps downstream.

In [ ]:
train_path = OUTPUT_DIR / 'train.csv'
test_path = OUTPUT_DIR / 'test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print('Saved:', train_path.resolve())
print('Saved:', test_path.resolve())